# Redrob Hackathon — Candidate Ranking Demo Sandbox

**Team:** Mysterio  
**Repo:** https://github.com/aditya-singh2005/Redrob-Hackathon-AI-Recruitment-Engine

This notebook runs the **full ranking pipeline end-to-end with zero manual steps**.
Everything is pulled from GitHub automatically — no uploads, no Drive access needed.

- Clones the repo and loads scripts automatically
- Uses the pre-bundled 100-candidate sample from the repo
- Runs feature extraction + embedding + ranking entirely on CPU
- Prints the ranked output and saves a CSV

**Just click Runtime → Run all. Nothing else needed.**

In [ ]:
# Cell 1 — Install dependencies
!pip install -q sentence-transformers pandas pyarrow scikit-learn tqdm

In [ ]:
# Cell 2 — Clone repo and set up folder structure
import os, shutil

# Clone the full repo
!git clone https://github.com/aditya-singh2005/Redrob-Hackathon-AI-Recruitment-Engine /content/redrob-repo

# Create working directories
os.makedirs('/content/redrob/data', exist_ok=True)
os.makedirs('/content/redrob/artifacts', exist_ok=True)
os.makedirs('/content/redrob/outputs', exist_ok=True)
os.makedirs('/content/redrob/scripts', exist_ok=True)

# Copy scripts from repo
for f in ['01_parse_jd.py', '02_feature_gen.py', '03_rank.py']:
    shutil.copy(f'/content/redrob-repo/scripts/{f}', f'/content/redrob/scripts/{f}')

# Copy the pre-bundled 100-candidate sample directly from repo — no upload needed
shutil.copy(
    '/content/redrob-repo/data/sample_candidates_100.jsonl',
    '/content/redrob/data/candidates.jsonl'
)

lines = sum(1 for _ in open('/content/redrob/data/candidates.jsonl'))
print(f'Setup complete ✓')
print(f'Scripts loaded: 3')
print(f'Candidates loaded: {lines}')

In [ ]:
# Cell 3 — Parse JD into structured requirements (instant)
%cd /content/redrob
!python scripts/01_parse_jd.py

In [ ]:
# Cell 4 — Extract features + embed candidate profiles
# On 100 candidates this takes ~30 seconds on CPU
!python scripts/02_feature_gen.py

In [ ]:
# Cell 5 — Rank candidates (CPU only, no API calls, no GPU)
# NOTE: with only 100 candidates the output will have fewer than 100 rows
# since the script picks the top-100 from whatever pool it receives.
# In production this runs against 100K candidates and outputs exactly 100.
!python scripts/03_rank.py --out outputs/sandbox_output.csv

In [ ]:
# Cell 6 — Display ranked results
import pandas as pd

df = pd.read_csv('/content/redrob/outputs/sandbox_output.csv')
print(f'\nRanked {len(df)} candidates from the 100-candidate sample')
print(f'Score range: {df["score"].min():.4f} – {df["score"].max():.4f}')
print(f'\nTop 10 ranked candidates:')
print('=' * 80)
for _, row in df.head(10).iterrows():
    print(f"Rank {int(row['rank']):3d} | {row['candidate_id']} | score={row['score']:.4f}")
    print(f"         {row['reasoning']}")
    print()

In [ ]:
# Cell 7 — Download the ranked CSV
from google.colab import files
files.download('/content/redrob/outputs/sandbox_output.csv')
print('Download triggered ✓')